In [1]:
import os
import random
import statistics
import re
import html
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
!pip install nltk
!pip install pymorphy3
!pip install stanza
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import stanza
from nltk.probability import FreqDist
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stopwords_ru = stopwords.words('russian')
from wordcloud import WordCloud

import pymorphy3
from pymorphy3 import MorphAnalyzer

morph = MorphAnalyzer()

from nltk import BigramCollocationFinder
from nltk.collocations import BigramAssocMeasures
from IPython.display import display
from pprint import pprint
# from tqdm import tqdm
from collections import Counter
! pip install PyMystem3
from pymystem3 import Mystem
morph = Mystem()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 28.1 MB/s eta 0:00:00


Installing mystem to /root/.local/bin/mystem from http://download.cdn.yandex.net/mystem/mystem-3.1-linux-64bit.tar.gz


In [2]:
stanza.download("ru")
nlp_stanza = stanza.Pipeline(lang="ru", processors="tokenize, pos, lemma")

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: ru (Russian) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/ru/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: ru (Russian):
| Processor | Package            |
----------------------------------
| tokenize  | syntagrus          |
| pos       | syntagrus_charlm   |
| lemma     | syntagrus_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


In [ ]:
# Установив все необходимые пакеты для данного корпуса, устанавливаем директорию, проверяем длину корпуса и файла
# 1. Извлечение данных и предобработка

In [6]:
file = os.listdir()[1:len(os.listdir())-1]

In [7]:
len(file)

1

In [ ]:
# Загрузка данных

In [8]:
text = "/content/Experts.txt"
len(text)

20

In [ ]:
# Очищаем текст, учитывая, что данные - научные публикации

In [9]:
text = "/content/Experts.txt"
with open(text, encoding='utf-8') as txt:
  text = txt.read()
  text = text.lower() # конвертируем весь текст в нижний регистр
  text = re.sub(r'[\r\n]+', ' ', text) # заменяем разрывы строк (переход на новую строку или абзац) на пробел
  text = re.sub(r'\xa0',' ', text)
  text = re.sub(r'\d+', '', text)
  text = re.sub(r'(?!AI)[^а-яА-ЯёЁьъ -]', ' ', text) # заменяем на пробел все символы, не относящиеся к буквам русского алфавита и не являющиеся пробелом либо дефисом
  text = re.sub(r'удк.*\n','',text)
  text = re.sub(r'\n|\t',' ',text)
  text = re.sub(r'рис.\s{0,1}[0-9]{1,2}',' ',text)
  text = re.sub(r'(список|источник)\s*литературы.*','',text)
  text_no_stop = ' '.join([token for token in word_tokenize(text) if token not in stopwords_ru]) # Что происходит в этой строке?
print("Количество токенов до обработки:\t", len(text.split()))
print("Количество неуникальных токенов после удаления стоп-слов:\t", len(text_no_stop.split()))
print("Количество уникальных токенов после удаления стоп-слов:\t", len(set(text_no_stop.split())))

Количество токенов до обработки:	 67872
Количество неуникальных токенов после удаления стоп-слов:	 53109
Количество уникальных токенов после удаления стоп-слов:	 14675


In [10]:
len(text)

766069

In [ ]:
# Извлекаем метаданные и составляем датафрейм на примере одной научной публикации, дополнительно очистив ее

In [12]:
with open('/content/Experts.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

In [13]:
def extract_metadata(raw_text):
    source = re.search(r'Вестник Евразийской науки|Современные научные исследования', raw_text)
    date = re.search(r'202[3-6]', raw_text)
    authors = re.findall(r'([А-ЯЁ][а-яё]+ [А-ЯЁ]\. ?[А-ЯЁ]\.)', raw_text)
    text = re.sub(r'Вестник Евразийской науки|Современные научные исследования|202[3-6]|[А-ЯЁ][а-яё]+ [А-ЯЁ]\. ?[А-ЯЁ]\.', '', raw_text)
    return {
        'Источник': source.group(0) if source else 'Неизвестно',
        'Дата': date.group(0) if date else 'Неизвестно',
        'Автор': ', '.join(authors) if authors else 'Неизвестно',
        'Текст': text.strip()
    }

metadata = extract_metadata(raw_text)

In [15]:
df = pd.DataFrame([metadata])
df.to_csv('metadata.csv', index=False)

In [18]:
nltk.download('punkt')
stanza.download('ru')
nlp = stanza.Pipeline('ru')
morph = MorphAnalyzer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: ru (Russian) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/ru/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: ru (Russian):
| Processor | Package            |
----------------------------------
| tokenize  | syntagrus          |
| pos       | syntagrus_charlm   |
| lemma     | syntagrus_nocharlm |
| depparse  | syntagrus_charlm   |
| ner       | wikiner            |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


In [ ]:
# 2. Токенизация и лемматизация

In [25]:
def lemmatize_with_stanza(text):
    doc = nlp(text)
    return [word.lemma for sent in doc.sentences for word in sent.words]

In [ ]:
# 3. Анализ по ключевым словам

In [ ]:
keywords = ['искусственный интеллект', 'ии', 'ai', 'нейросети']
counts_total = {kw: 0 for kw in keywords}
counts_unique = {kw: set() for kw in keywords}
all_lemmas = []
unique_lemmas = set()
lemma_records = []

for idx, row in metadata_df.iterrows():
    text = row['Текст_очищенный']
    tokens = nltk.word_tokenize(text)
    lemmas_pymorphy = lemmatize_with_pymorphy(tokens)
    lemmas_stanza = lemmatize_with_stanza(text)
    lemmas = list(set(lemmas_pymorphy + lemmas_stanza))
    all_lemmas.extend(lemmas)
    unique_lemmas.update(lemmas)

    def extract_context_lemmas(tokens, lemmas_pymorphy, kw):
        kw_tokens = kw.split()
        kw_len = len(kw_tokens)
        context_lemmas = set()
        for i in range(len(tokens) - kw_len + 1):
            if tokens[i:i+kw_len] == kw_tokens:
                start = max(0, i-5)
                end = min(len(lemmas_pymorphy), i+kw_len+5)
                context_lemmas.update(lemmas_pymorphy[start:end])
        return context_lemmas

    for kw in keywords:
        context_lemmas = extract_context_lemmas(tokens, lemmas_pymorphy, kw)
        counts_total[kw] += len(context_lemmas)
        counts_unique[kw].update(context_lemmas)
        for lemma in context_lemmas:
            lemma_records.append({'keyword': kw, 'lemma': lemma})

In [ ]:
# 4. Сохранение результатов

In [ ]:
lemma_df = pd.DataFrame(lemma_records)
counts_unique_list = {kw: len(vals) for kw, vals in counts_unique.items()}
counts_total_list = counts_total

lemma_counts = Counter(all_lemmas)
top_lemmas_df = pd.DataFrame(lemma_counts.most_common(20), columns=['Лемма', 'Частота'])
top_lemmas_df.to_csv('top_lemmas.csv', index=False, encoding='utf-8')

counts_df = pd.DataFrame([counts_total_list, counts_unique_list], index=['Всего лемм', 'Уникальных лемм']).T
counts_df.to_csv('lemma_counts.csv', index=True, encoding='utf-8')
lemma_df.to_csv('lemma_keywords.csv', index=False, encoding='utf-8')
metadata_df.to_csv('metadata.csv', index=False, encoding='utf-8')

In [ ]:
# 5. Коллокации

In [ ]:
bigram_measures = nltk.collocations.BigramAssocMeasures()
finder_bi = nltk.BigramCollocationFinder.from_words(all_lemmas)
finder_bi.apply_freq_filter(3)
collocations_bi = finder_bi.nbest(bigram_measures.pmi, 20)
collocations_bi_df = pd.DataFrame(collocations_bi, columns=['Коллокация 1', 'Коллокация 2'])
collocations_bi_df.to_csv('collocations_bigrams.csv', index=False, encoding='utf-8')

In [ ]:
# 6. NER

In [ ]:
ner_records = []
for idx, row in metadata_df.iterrows():
    doc = nlp(row['Текст_очищенный'])
    for sent in doc.sentences:
        for ent in sent.ents:
            ner_records.append({
                'Текст': row['Текст'],
                'Сущность': ent.text,
                'Тип': ent.type,
                'Начало': ent.start_char,
                'Конец': ent.end_char,
            })
ner_df = pd.DataFrame(ner_records)
ner_df.to_csv('ner_entities.csv', index=False, encoding='utf-8')

In [ ]:
# 7. Статистический анализ (хи-квадрат)

In [ ]:

# Пример: сравнение частоты "эффективен" в документах с ИИ и без ИИ (по ключевым словам)
docs_with_ai = metadata_df[metadata_df['Текст_очищенный'].str.contains('|'.join(keywords))]
docs_without_ai = metadata_df[~metadata_df['Текст_очищенный'].str.contains('|'.join(keywords))]
count_eff_ai = docs_with_ai['Текст_очищенный'].str.count('эффективн').sum()
count_not_eff_ai = docs_with_ai['Текст_очищенный'].str.len().sum() - count_eff_ai
count_eff_noai = docs_without_ai['Текст_очищенный'].str.count('эффективн').sum()
count_not_eff_noai = docs_without_ai['Текст_очищенный'].str.len().sum() - count_eff_noai

obs = [[count_eff_ai, count_not_eff_ai], [count_eff_noai, count_not_eff_noai]]
chi2, p, dof, expected = stats.chi2_contingency(obs)
print(f'Хи-квадрат: {chi2:.2f}, p-value: {p:.3f}')

In [ ]:
# 8. Выводы и гипотезы

In [ ]:
effective_terms = ['эффективн', 'результат', 'повышени', 'рост', 'улучшени', 'успех', 'достижен']
effective_count = sum(lemma_counts.get(term, 0) for term in effective_terms)
total_count = sum(lemma_counts.values())
effective_share = effective_count / total_count if total_count > 0 else 0

print(f'Доля лемм с положительной коннотацией: {effective_share:.2%}')
if effective_share >= 0.1:
    print("В корпусе преобладают положительные оценки эффективности ИИ в маркетинге.")
elif effective_share >= 0.05:
    print("В корпусе смешанные оценки эффективности ИИ в маркетинге.")
else:
    print("В корпусе преобладают нейтральные или негативные оценки эффективности ИИ в маркетинге.")